In [33]:
import os
import json
import base64
import requests
import pandas as pd
import time
import re 
from pathlib import Path
from PIL import Image
from io import BytesIO
from openai import OpenAI

# --- Configuration ---
OPENROUTER_API_KEY = "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"
MODEL_NAME = "mistralai/mistral-small-3.1-24b-instruct:free" 
MAX_RETRIES = 3

NGROK_URL = "https://3325-194-27-149-203.ngrok-free.app/v1"
# MODELS_TO_TEST = ["llava:7b", "qwen2.5vl:7b", "qwen3-vl:8b"]
# MODELS_TO_TEST = ["llava:7b"]
# MODELS_TO_TEST = ["qwen2.5vl:7b"]
MODELS_TO_TEST = ["qwen3-vl:8b"]

In [ ]:
def encode_image_from_bytes(byte_data):
    """Converts raw bytes to a base64 encoded string."""
    try:
        base64_str = base64.b64encode(byte_data).decode('utf-8')
        return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None

def query_vision_model(prompt, base64_image, max_retries=MAX_RETRIES):
    """Sends the image and prompt to the VLM, forcing a short answer and tracking execution time."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    
    strict_prompt = f"""{prompt}

You are an expert mathematician and visual analyst. Answer the user's question using only the provided image and text.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the image").
2. If the answer is a letter matching a multiple choice option (A, B, C, D), simply output the letter.
3. Output equations or numbers directly.
"""
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": strict_prompt},
                    {"type": "image_url", "image_url": {"url": base64_image}}
                ]
            }
        ]
    }
    
    start_time = time.time()
    for attempt in range(max_retries):
        try:
            response = requests.post(
                url="https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=120
            )
            response.raise_for_status()
            result_text = response.json()['choices'][0]['message']['content'].strip()
            duration = time.time() - start_time
            return result_text, duration
        except requests.exceptions.Timeout:
            print(f"   ⏳ Attempt {attempt + 1}/{max_retries}: Request timed out.")
        except requests.exceptions.RequestException as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    duration = time.time() - start_time
    return "ERROR_MAX_RETRIES_REACHED", duration

def evaluate_math(ans, gt):
    """Evaluates if the model answer matches the ground truth."""
    ans_clean = str(ans).lower().replace(" ", "")
    gt_clean = str(gt).lower().replace(" ", "")
    
    # Direct exact match
    if ans_clean == gt_clean:
        return True
    
    # Substring match (in case the model is slightly verbose despite instructions)
    if gt_clean in ans_clean:
        return True
        
    # Fallback structure
    return False

def run_evaluation(df, limit=None):
    print(f"🚀 Starting MathVision inference using {MODEL_NAME}...")
    
    safe_model_name = MODEL_NAME.replace("/", "_").replace(":", "_")
    
    # Create output directory if it does not exist
    results_dir = Path("MathVisionResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"mathvision_{safe_model_name}_results.csv"
    
    processed_tasks = set()
    file_exists = output_csv.is_file()
    
    if file_exists:
        try:
            existing_df = pd.read_csv(output_csv)
            if not existing_df.empty and 'id' in existing_df.columns:
                processed_tasks = set(existing_df['id'].astype(str))
                print(f"🔄 Found existing CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
        except Exception as e:
            print(f"⚠️ Could not read existing CSV for progress tracking: {e}")
            
    count = 0
    for index, row in df.iterrows():
        if limit and count >= limit:
            break
            
        task_id = str(row['id'])
        if task_id in processed_tasks:
            continue
            
        level = str(row.get('level', 'N/A'))
        subject = str(row.get('subject', 'N/A'))
        image_name = str(row.get('image', 'N/A'))
        image_name = str(row.get('image', 'N/A'))
        query = str(row['question'])
        ground_truth = str(row['answer']).strip()
        
        print(f"\nProcessing Task: {task_id} | Subject: {subject} | Lvl: {level}")
        
        img_data = None
        base64_img = None
        
        if 'decoded_image' in row and isinstance(row['decoded_image'], dict):
            img_data = row['decoded_image'].get('bytes')
            
        if img_data:
            base64_img = encode_image_from_bytes(img_data)
        
        if not base64_img:
            print("   ⚠️ Skipping due to image parsing error.")
            continue
            
        model_answer, duration = query_vision_model(query, base64_img)
        evaluation_passed = evaluate_math(model_answer, ground_truth)
        
         # Save to CSV
            result_df = pd.DataFrame([{
                'id': task_id,
                'level': level,
                'subject': subject,
                'image' : img_name,
                'question': query,
                'ground_truth': ground_truth,
                'model_answer': model_answer,
                'evaluation_passed': evaluation_passed,
                'run_time': round(duration, 3),
            }])
        
        result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
        file_exists = True
        
        processed_tasks.add(task_id)
        count += 1
        
        eval_mark = "✅" if evaluation_passed else "❌"
        print(f"   {eval_mark} Evaluation | Time: {duration:.2f}s | Reply: '{model_answer}'")
        time.sleep(3) # API rate limit pause

if 'df' in globals():
    # Uncomment the limit parameter to test on a subset first
    run_evaluation(df, limit=2)
else:
    print("❌ ‘df’ memory object missing. Please run Cell 1 first to load the dataset.")


In [34]:
# Standardize Ngrok connection
client = OpenAI(
    base_url=NGROK_URL,
    api_key="sk-no-key-required"
)

def encode_image_from_bytes(byte_data):
    """Converts raw bytes to a base64 encoded string."""
    try:
        base64_str = base64.b64encode(byte_data).decode('utf-8')
        return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None

def query_vision_model(prompt, base64_image, model_name, max_retries=MAX_RETRIES):
    """Sends the image and prompt directly via the OpenAI package to Ngrok backend."""
    strict_prompt = f"""{prompt}

You are an expert mathematician and visual analyst. Answer the user's question using only the provided image and text.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the image").
2. If the answer is a letter matching a multiple choice option (A, B, C, D), simply output the letter.
3. Output equations or numbers directly.
"""
    
    messages = [
         {
            "role": "user",
            "content": [
                {"type": "text", "text": strict_prompt},
                {"type": "image_url", "image_url": {"url": base64_image}}
            ]
        }
    ]
    
    start_time = time.time()
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                stream=False,
                temperature=0.1
            )
            result_text = response.choices[0].message.content.strip()
            duration = time.time() - start_time
            return result_text, duration
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    duration = time.time() - start_time
    return "ERROR_MAX_RETRIES_REACHED", duration

def evaluate_math(ans, gt):
    """Evaluates if the model answer matches the ground truth."""
    ans_clean = str(ans).lower().replace(" ", "")
    gt_clean = str(gt).lower().replace(" ", "")
    
    if ans_clean == gt_clean:
        return True
    if gt_clean in ans_clean:
        return True
        
    return False

def run_evaluation(df, limit=None):
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 Starting MathVision NGROK inference using {model_name}...")
        
        safe_model_name = model_name.replace("/", "_").replace(":", "_")
        
        results_dir = Path("MathVisionResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"mathvision_ngrok_{safe_model_name}_results.csv"
        
        processed_tasks = set()
        file_exists = output_csv.is_file()
        
        if file_exists:
            try:
                existing_df = pd.read_csv(output_csv)
                if not existing_df.empty and 'id' in existing_df.columns:
                    processed_tasks = set(existing_df['id'].astype(str))
                    print(f"🔄 Found existing CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
            except Exception as e:
                print(f"⚠️ Could not read existing CSV for progress tracking: {e}")
                
        count = 0
        for index, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break
                
            task_id = str(row['id'])
            if task_id in processed_tasks:
                continue
                
            level = str(row.get('level', 'N/A'))
            subject = str(row.get('subject', 'N/A'))
            image_name = str(row.get('image', 'N/A'))
            image_name = str(row.get('image', 'N/A'))
            query = str(row['question'])
            ground_truth = str(row['answer']).strip()
            
            print(f"\nProcessing Task: {task_id} | Subject: {subject} | Lvl: {level}")
            
            img_data = None
            base64_img = None
            
            if 'decoded_image' in row and isinstance(row['decoded_image'], dict):
                img_data = row['decoded_image'].get('bytes')
                
            if img_data:
                base64_img = encode_image_from_bytes(img_data)
            
            if not base64_img:
                print("   ⚠️ Skipping due to image parsing error.")
                continue
                
            model_answer, duration = query_vision_model(query, base64_img, model_name)
            evaluation_passed = evaluate_math(model_answer, ground_truth)
            
            # Save to CSV
            result_df = pd.DataFrame([{
                'id': task_id,
                'level': level,
                'subject': subject,
                'image' : img_name,
                'question': query,
                'ground_truth': ground_truth,
                'model_answer': model_answer,
                'evaluation_passed': evaluation_passed,
                'run_time': round(duration, 3),
            }])
            
            result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
            file_exists = True
            
            processed_tasks.add(task_id)
            count += 1
            
            eval_mark = "✅" if evaluation_passed else "❌"
            print(f"   {eval_mark} Evaluation | Time: {duration:.2f}s | Reply: '{model_answer}'")
            time.sleep(3) # API rate limit pause

if 'df' in globals():
    # Uncomment the limit parameter to test on a subset first
    run_evaluation(df, limit=10)
else:
    print("❌ ‘df’ memory object missing. Please run Cell 1 first to load the dataset.")



🚀 Starting MathVision NGROK inference using qwen3-vl:8b...
🔄 Found existing CSV! Resuming... Skipping 2 already processed tasks.

Processing Task: 3 | Subject: metric geometry - length | Lvl: 1


KeyboardInterrupt: 